# GA4 Ecommerce Funnel Data Prep


Evelina Ramoskaite

### Extraction

In [2]:
# Import libraries
from google.cloud import bigquery # reference readme file for setup/authentication instructions
import pandas as pd


In [3]:
# Specifying Project ID 
PROJECT_ID = 'bigquery-public-evelinar'
client = bigquery.Client(project=PROJECT_ID)

In [4]:
# Set Start and End Date
# Format: YYYYMMDD
START_DATE = '20210101'
END_DATE = '20210131'


In [5]:
# Extracting data from Bigquery 
with open('checkout_funnel_daily.sql') as f:
     query = f.read().format(start_date=START_DATE, end_date=END_DATE)
df = client.query(query).to_dataframe()


In [261]:
print(df.shape)
df.head(40)

(238073, 12)


,event_date,event_name,user_pseudo_id,session_id,event_ts,country,state,device_category,traffic_medium,traffic_source,transaction_id,revenue
0,20210124,session_start,1008607.7032882869,521445891,2021-01-24 10:19:49.218426+00:00,United States,Ohio,mobile,organic,google,NaN,NaN
1,20210124,view_item,1027865.2921216251,6122537537,2021-01-24 20:47:05.204087+00:00,United States,Pennsylvania,mobile,organic,google,(not set),NaN
2,20210124,session_start,1027865.2921216251,6122537537,2021-01-24 20:43:56.274718+00:00,United States,Pennsylvania,mobile,organic,google,NaN,NaN
3,20210124,session_start,1027865.2921216251,8887175570,2021-01-24 22:39:01.029399+00:00,United States,Pennsylvania,mobile,referral,shop.googlemerchandisestore.com,NaN,NaN
4,20210124,session_start,1043145.0716229192,7515384017,2021-01-24 23:58:49.575521+00:00,United States,Virginia,mobile,organic,google,NaN,NaN
5,20210124,session_start,1044510.5873644104,6078712742,2021-01-24 18:14:07.911914+00:00,United States,Florida,desktop,(none),(direct),NaN,NaN
6,20210124,session_start,1048584.7594502159,9109808764,2021-01-24 13:34:14.674744+00:00,Sweden,Vastra Gotaland County,desktop,<Other>,<Other>,NaN,NaN
7,20210124,view_item,1080463.7425293909,612924697,2021-01-24 12:12:24.028087+00:00,United States,Connecticut,desktop,(none),(direct),(not set),NaN
8,20210124,session_start,1080463.7425293909,612924697,2021-01-24 12:10:18.088711+00:00,United States,Connecticut,desktop,(none),(direct),NaN,NaN
9,20210124,add_shipping_info,1080463.7425293909,612924697,2021-01-24 12:15:40.226652+00:00,United States,Connecticut,desktop,(none),(direct),(not set),NaN


## Cleaning

In [6]:
# Format Date
df['event_date'] = pd.to_datetime(df['event_date'], format = '%Y%m%d')

In [7]:
# creating a unique session key
# session_id by itself is just time-based
df['session_key'] = df['user_pseudo_id'] + '-' + df['session_id'].astype(str)
df = df.drop(columns=['user_pseudo_id', 'session_id'])

In [8]:
# Dropping Duplicate events, if any
print(len(df))
df = df.drop_duplicates(subset= ['session_key','event_name','event_ts'])
print(len(df))

238073
238073


In [9]:
#Checking Missing Values
df.isna().sum()

event_date              0
event_name              0
event_ts                0
country                 0
state                   0
device_category         0
traffic_medium          0
traffic_source          0
transaction_id     116549
revenue            237169
session_key             0
dtype: int64

In [10]:
df.head(50)

,event_date,event_name,event_ts,country,state,device_category,traffic_medium,traffic_source,transaction_id,revenue,session_key
0,2021-01-01,session_start,2021-01-01 02:31:58.563156+00:00,France,Auvergne-Rhone-Alpes,desktop,organic,google,NaN,NaN,1003046.9452926974-3209612510
1,2021-01-01,view_item,2021-01-01 16:17:24.961359+00:00,India,Maharashtra,mobile,<Other>,<Other>,(not set),NaN,1023282.4639847710-7473279052
2,2021-01-01,session_start,2021-01-01 16:15:31.708459+00:00,India,Maharashtra,mobile,<Other>,<Other>,NaN,NaN,1023282.4639847710-7473279052
3,2021-01-01,add_shipping_info,2021-01-01 16:26:27.293518+00:00,India,Maharashtra,mobile,<Other>,<Other>,(not set),NaN,1023282.4639847710-7473279052
4,2021-01-01,view_item,2021-01-01 16:17:03.025952+00:00,India,Maharashtra,mobile,<Other>,<Other>,(not set),NaN,1023282.4639847710-7473279052
5,2021-01-01,view_item,2021-01-01 16:25:20.637924+00:00,India,Maharashtra,mobile,<Other>,<Other>,(not set),NaN,1023282.4639847710-7473279052
6,2021-01-01,add_shipping_info,2021-01-01 16:27:31.697610+00:00,India,Maharashtra,mobile,<Other>,<Other>,(not set),NaN,1023282.4639847710-7473279052
7,2021-01-01,view_item,2021-01-01 16:24:46.227228+00:00,India,Maharashtra,mobile,<Other>,<Other>,(not set),NaN,1023282.4639847710-7473279052
8,2021-01-01,session_start,2021-01-01 15:48:32.499719+00:00,United States,Wisconsin,desktop,referral,<Other>,NaN,NaN,1026043.4481623879-5588834374
9,2021-01-01,session_start,2021-01-01 12:25:51.286460+00:00,United States,Wisconsin,desktop,cpc,google,NaN,NaN,1026043.4481623879-9083794581


In [267]:
# 2. Get unique values 
for c in ['traffic_source','traffic_medium','device_category']:
    print(c)
    print(df[c].unique().tolist())

traffic_source
['google', 'shop.googlemerchandisestore.com', '(direct)', '<Other>', '(data deleted)']
traffic_medium
['organic', 'referral', '(none)', '<Other>', 'cpc', '(data deleted)']
device_category
['mobile', 'desktop', 'tablet']


In [20]:
# defining channel names
def channel(row):
    src = row['traffic_source']
    med = row['traffic_medium']

    if src == 'shop.googlemerchandisestore.com':
        return 'Unattributed'
    if med == 'organic':
        return 'Organic Search'
    if med == 'cpc':
        return 'Paid Search'
    if med == 'referral':
        return 'Referral'
    if src == '(direct)':
        return 'Direct'
    if '(data deleted)' in (src, med):
        return 'Unattributed'
    return 'Unattributed'

df['channel'] = df.apply(channel, axis=1)

### Funnel Data Prep

In [21]:
FUNNEL_STEPS = [
    'session_start',
    'view_item',
    'add_to_cart',
    'begin_checkout',
    'add_shipping_info',
    'add_payment_info',
    'purchase',
]
LABELS = {
    'session_start': 'Session',
    'view_item': 'Product View',
    'add_to_cart': 'Add to Cart',
    'begin_checkout': 'Checkout Started',
    'add_shipping_info': 'Shipping Info',
    'add_payment_info': 'Payment Info',
    'purchase': 'Purchase',
}

STEP_ORDER = {
    'session_start': 1,
    'view_item': 2,
    'add_to_cart': 3,
    'begin_checkout': 4,
    'add_shipping_info': 5,
    'add_payment_info': 6,
    'purchase': 7,
}

In [22]:
df['step'] = df['event_name'].map(LABELS)
df['step_order'] = df['event_name'].map(STEP_ORDER)

In [23]:
# Aggregating funnel data. 
funnel = df.groupby(['event_date','device_category','step','step_order'])['session_key'].nunique().reset_index(name = 'sessions')

In [24]:
#Check
funnel.groupby('step_order')['sessions'].sum()

step_order
1    116514
2     23153
3      4548
4      2161
5      2161
6      1563
7      1116
Name: sessions, dtype: int64

### Purchase Data Prep

In [25]:
# Aggregating purchase data
purchases = df[df['event_name'] == 'purchase'].drop_duplicates(subset= ['transaction_id','session_key'])
pur_df = purchases.groupby(['event_date','country','device_category','channel'])['revenue'].sum().reset_index()

In [26]:
purchases.head(5)

,event_date,event_name,event_ts,country,state,device_category,traffic_medium,traffic_source,transaction_id,revenue,session_key,channel,step,step_order
2840,2021-01-01,purchase,2021-01-01 18:47:04.715080+00:00,(not set),(not set),mobile,organic,google,887131,22.0,3351786.5083077372-8044807491,Organic Search,Purchase,7
2898,2021-01-01,purchase,2021-01-01 03:29:50.052499+00:00,Israel,Center District,mobile,(none),(direct),722217,20.0,40650638.8176789956-3410418880,Direct,Purchase,7
2934,2021-01-01,purchase,2021-01-01 04:55:33.799172+00:00,United States,Indiana,desktop,(data deleted),(data deleted),146188,14.0,87047667.1042912591-5358392223,Unattributed,Purchase,7
2937,2021-01-01,purchase,2021-01-01 04:49:56.446915+00:00,United States,Indiana,desktop,(data deleted),(data deleted),966325,44.0,87047667.1042912591-5358392223,Unattributed,Purchase,7
2963,2021-01-01,purchase,2021-01-01 05:53:02.021197+00:00,India,Karnataka,mobile,(none),(direct),479241,52.0,68546038.8820428933-5425867143,Direct,Purchase,7


### Export

In [27]:
# Save to CSV
funnel.to_csv('Funnel.csv', index=False)
pur_df.to_csv('Purchases.csv',index=False)